In [10]:
QUERY_AUDIO_PATH = "C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav"

FILENAME= QUERY_AUDIO_PATH.split("/")[-1].split(".")[0]
FILENAME

'dark-distorted-hard-trap-bass_F_minor'

In [12]:
import asyncio
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from qdrant_client import QdrantClient
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("../.."))
from config.settings import settings
from rag.clap.model_handler import create_clap_model

QDRANT_COLLECTION = settings.QDRANT_ENRICHED_COLLECTION_NAME
BACKGROUND_SAMPLES = 400
TOP_K = 10
FILENAME = os.path.basename(QUERY_AUDIO_PATH)

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

async def print_audio_retrieval_table(search_res):
    table_data = []

    for rank, hit in enumerate(search_res.points, start=1):
        payload = hit.payload or {}
        ai_label = payload.get("ai_label", "N/A")
        orig_label = payload.get("original_label") or payload.get("original_filename", "N/A")
        tags = payload.get("ai_tags", [])
        tags_str = ", ".join(tags[:5]) if isinstance(tags, list) else str(tags)
        stored_quality = payload.get("clap_score", 0.0)

        table_data.append({
            "Rank": rank,
            "Similarity Score": hit.score,
            "AI Description": ai_label,
            "Original Label": orig_label,
            "Tags": tags_str,
            "Label Quality": stored_quality
        })

    df = pd.DataFrame(table_data)

    csv_path = os.path.join(RESULTS_DIR, f"audio_retrieval_table_{FILENAME}.csv")
    df.to_csv(csv_path, index=False)
    print(f"Tabella salvata in: {csv_path}")

    print(f"TOP-{TOP_K} FILE ACUSTICAMENTE SIMILI A: {QUERY_AUDIO_PATH}")
    print("="*80)

    pd.set_option('display.max_colwidth', None)
    display(df.style.background_gradient(subset=['Similarity Score'], cmap='Blues')
                    .background_gradient(subset=['Label Quality'], cmap='Greens')
                    .format({'Similarity Score': '{:.4f}', 'Label Quality': '{:.4f}'}))

async def visualize_audio_semantic_map():
    if not os.path.exists(QUERY_AUDIO_PATH):
        print(f"ERRORE: Il file '{QUERY_AUDIO_PATH}' non è stato trovato nella cartella corrente.")
        return

    print(f"--- ANALISI RETRIEVAL AUDIO: '{QUERY_AUDIO_PATH}' ---")

    client_q = QdrantClient(host=settings.QDRANT_CONNECTION_HOST, port=settings.QDRANT_PORT)
    clap = create_clap_model()

    print("Calcolo embedding audio input...")
    query_vector = clap.get_audio_embedding([QUERY_AUDIO_PATH])[0]

    print("Esecuzione ricerca vettoriale (Audio-to-Audio)...")
    search_res = client_q.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        using="audio_vector",
        limit=TOP_K,
        with_payload=True,
        with_vectors=["audio_vector"]
    )

    print("Recupero contesto di fondo...")
    scroll_res, _ = client_q.scroll(
        collection_name=QDRANT_COLLECTION,
        limit=BACKGROUND_SAMPLES,
        with_payload=True,
        with_vectors=["audio_vector"]
    )

    vectors = []
    meta = []
    vectors.append(query_vector)
    meta.append({
        "name": "INPUT AUDIO",
        "desc": f"File: {QUERY_AUDIO_PATH}",
        "score": 1.0,
        "type": "query"
    })

    result_indices = []
    for i, hit in enumerate(search_res.points):
        vec = hit.vector.get("audio_vector") if isinstance(hit.vector, dict) else hit.vector
        if vec is None: continue

        vectors.append(vec)
        payload = hit.payload or {}

        fname = payload.get("original_filename", "Unknown")
        label = payload.get("label", "")
        tags = payload.get("ai_tags", [])
        desc_text = f"Label: {label}<br>Tags: {', '.join(tags[:3])}"

        meta.append({
            "name": fname,
            "desc": desc_text,
            "score": hit.score,
            "type": "result"
        })
        result_indices.append(len(vectors) - 1)

    for hit in scroll_res:
        if hit.id in [r.id for r in search_res.points]:
            continue

        vec = hit.vector.get("audio_vector") if isinstance(hit.vector, dict) else hit.vector
        if vec is None: continue

        vectors.append(vec)
        payload = hit.payload or {}
        meta.append({
            "name": "Background",
            "desc": payload.get("label", "No Label"),
            "score": 0,
            "type": "noise"
        })

    print("Proiezione 2D (PCA + t-SNE)...")
    X = np.array(vectors)

    pca = PCA(n_components=min(50, len(X)))
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=min(30, len(X)-1), random_state=42, init='pca', learning_rate='auto')
    X_2d = tsne.fit_transform(X_pca)

    fig = go.Figure()

    q_x, q_y = X_2d[0]

    for idx in result_indices:
        r_x, r_y = X_2d[idx]
        score = meta[idx]['score']

        fig.add_trace(go.Scatter(
            x=[q_x, r_x], y=[q_y, r_y],
            mode='lines',
            line=dict(
                color='rgba(0, 0, 255, 0.4)',
                width=score * 5
            ),
            hoverinfo='none',
            showlegend=False
        ))

    bg_indices = [i for i, m in enumerate(meta) if m['type'] == 'noise']
    bg_x = X_2d[bg_indices, 0]
    bg_y = X_2d[bg_indices, 1]
    bg_text = [meta[i]['desc'] for i in bg_indices]

    fig.add_trace(go.Scatter(
        x=bg_x, y=bg_y,
        mode='markers',
        marker=dict(size=6, color='#bdc3c7', opacity=0.4),
        text=bg_text,
        name='Context (Database)',
        hoverinfo='text'
    ))

    res_x = [X_2d[i][0] for i in result_indices]
    res_y = [X_2d[i][1] for i in result_indices]
    res_scores = [meta[i]['score'] for i in result_indices]
    res_text = [f"<b>{meta[i]['name']}</b><br>Score: {meta[i]['score']:.3f}<br>{meta[i]['desc']}" for i in result_indices]

    fig.add_trace(go.Scatter(
        x=res_x, y=res_y,
        mode='markers',
        marker=dict(
            size=15,
            color=res_scores,
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title="Similarity Score"),
            line=dict(color='white', width=2)
        ),
        text=res_text,
        name='Similar Sounds',
        hoverinfo='text'
    ))

    fig.add_trace(go.Scatter(
        x=[q_x], y=[q_y],
        mode='markers',
        marker=dict(size=25, color='red', symbol='star', line=dict(color='black', width=1)),
        name='Input Audio File',
        hoverinfo='name'
    ))

    fig.update_layout(
        title=f"<b>Audio Retrieval Map</b><br>Input: {FILENAME}",
        width=1000,
        height=800,
        template="plotly_white",
        xaxis=dict(showgrid=False, zeroline=False, visible=False),
        yaxis=dict(showgrid=False, zeroline=False, visible=False),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )

    plot_path = os.path.join(RESULTS_DIR, f"audio_retrieval_map_{FILENAME}.html")
    fig.write_html(plot_path)
    print(f"Grafico salvato in: {plot_path}")

    await print_audio_retrieval_table(search_res=search_res)

    fig.show()

await visualize_audio_semantic_map()

--- ANALISI RETRIEVAL AUDIO: 'C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav' ---
Calcolo embedding audio input...
Esecuzione ricerca vettoriale (Audio-to-Audio)...
Recupero contesto di fondo...
Proiezione 2D (PCA + t-SNE)...
Grafico salvato in: ./results\audio_retrieval_map_dark-distorted-hard-trap-bass_F_minor.wav.html
Tabella salvata in: ./results\audio_retrieval_table_dark-distorted-hard-trap-bass_F_minor.wav.csv
TOP-10 FILE ACUSTICAMENTE SIMILI A: C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav


,Rank,Similarity Score,AI Description,Original Label,Tags,Label Quality
0,1,0.6485,"Bass-line with Clean, Dark, and Energetic Character",Dubstep Bass Line - Hard,"dynamic, clean, dark, dry, energetic",0.3183
1,2,0.6473,Bass-line with aggressive and dynamic slides,UK Drill 808 with Smooth Slides,"aggressive, dynamic, heavy, energetic",0.1598
2,3,0.6350,Analog 808 Bass-Line with Distorted Staccato Hits,808 Rap Type Bass - Line Staccato Hits,"cold, anxious, staccato, low, pulsating",0.4338
3,4,0.6213,Digital Synth Bass with Calm and Dreamy Texture,Calm Synth Bass Line,"ambient, chillout, soft, chilled, calm",0.3873
4,5,0.6200,Sub-bass with Pulsating Ambient Texture,Low Pulsating Bass,"dynamic, ambient, full, soft, echoing",0.4174
5,6,0.6174,"Analog Synth Bass with Hard, Dark, and Dry Texture",Phonk Bass Shot - Midnight Hit,"dynamic, hard, clean, analog, dark",0.2842
6,7,0.6152,"Sub-bass with flowing and heavy timbre, short one-shot articulation",Spring 808 Bass,"short, one shot, flowing, heavy",0.4029
7,8,0.6135,"Analog Synth Bass with Warm, Flowing Character",''Periord'' - Heavy Bass Progression,"short, one shot, warm, flowing",0.4829
8,9,0.6125,"Sub-bass with smooth, soft timbre",Smooth Sub Bass,"ambient, chillout, soft, chilled, calm",0.2571
9,10,0.6119,"Digital Synth Bass with Sharp, Spiky Texture and Heavy, Dark Ambience",Sharp Spikey Bass Line,"eerie, dynamic, gloomy, ambient, dark",0.1923
